# Can Weather Predict Air Pollution in Berlin?
### A Bayesian regression analysis of temperature, wind speed, and NO₂ at Station 221 Karl-Marx-Straße

This notebook follows the same section order as my PDF report.  
The main question is: **to what extent can daily weather conditions predict NO₂ concentrations in Berlin, and does adding model complexity improve predictions on unseen days?**

Code structure is adapted from:
- CS146 Session 7, `07a. Pre-class work.ipynb` (linear regression, prior/posterior checks)
- CS146 Session 8, `08a. Pre-class work.ipynb` (thinking about outliers and robust regression)
- CS146 Session 9, `09a. Pre-class work.ipynb` (polynomial regression)
- CS146 Session 10, `10a. Pre-class work.ipynb` (underfitting, overfitting, and model comparison)

## 1. Introduction

I use daily NO₂ measurements from one Berlin traffic-monitoring station and daily weather data for 2025.  
The models are predictive, not causal: a regression coefficient describes an association in this data set, not proof that changing temperature or wind would directly cause NO₂ to change.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pymc as pm
import arviz as az

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Reproducibility
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Match the 89% credible intervals used in the class workbooks
az.rcParams["stats.ci_prob"] = 0.89

## 2. Data Collection and Exploratory Analysis

**Air pollution:** Berliner Luftgütemessnetz, Station 221 Karl-Marx-Straße.  
The file contains daily values for PM10, PM2.5, NO₂, NO, and NOx. I only use NO₂ as the response variable.

**Weather:** Open-Meteo Historical Weather API.  
I use daily mean temperature at 2 m, precipitation sum, and maximum wind speed at 10 m.

The two sources are merged by date. The weather data are reanalysis estimates on a spatial grid, not measurements from an instrument attached to the pollution station.

In [ ]:
# Weather CSV: the first 3 rows are metadata
weather = pd.read_csv("berlin_weather_2025.csv", skiprows=3)

weather = weather.rename(columns={
    "time": "date",
    "temperature_2m_mean (°C)": "temp_mean",
    "precipitation_sum (mm)": "precip",
    "wind_speed_10m_max (km/h)": "wind_max"
})

weather["date"] = pd.to_datetime(weather["date"])
weather.head()

In [ ]:
# Pollution CSV: semicolon-separated, with 5 metadata/header rows above the daily data
# The filename says 2026, but the observations inside are for 2025.
pollution = pd.read_csv(
    "NO2_pollution_2025.csv",
    sep=";",
    skiprows=5,
    header=None,
    names=["date", "pm10", "pm25", "no2", "no", "nox"]
)

pollution["date"] = pd.to_datetime(pollution["date"], format="%d.%m.%Y")
pollution.head()

In [ ]:
# Merge the two files by date
df = pd.merge(pollution, weather, on="date", how="inner")

# Keep only variables that are relevant to this analysis
data = df[["date", "no2", "temp_mean", "precip", "wind_max"]].copy()

print("Rows:", len(data))
print("\nMissing values:")
print(data.isna().sum())
print("\nSummary statistics:")
display(data.describe())

In [ ]:
# Daily NO2 through the year
plt.figure(figsize=(11, 4))
plt.plot(data["date"], data["no2"])
plt.xlabel("Date")
plt.ylabel("Daily mean NO₂ (µg/m³)")
plt.title("Daily NO₂ concentration at Karl-Marx-Straße, Berlin, 2025")
plt.show()

In [ ]:
# Temperature vs NO2
plt.figure(figsize=(6, 4))
plt.scatter(data["temp_mean"], data["no2"], alpha=0.55)
plt.xlabel("Mean daily temperature (°C)")
plt.ylabel("Daily mean NO₂ (µg/m³)")
plt.title("Temperature and NO₂")
plt.show()

In [ ]:
# Wind vs NO2
plt.figure(figsize=(6, 4))
plt.scatter(data["wind_max"], data["no2"], alpha=0.55)
plt.xlabel("Maximum daily wind speed (km/h)")
plt.ylabel("Daily mean NO₂ (µg/m³)")
plt.title("Wind speed and NO₂")
plt.show()

In [ ]:
# Precipitation vs NO2
plt.figure(figsize=(6, 4))
plt.scatter(data["precip"], data["no2"], alpha=0.55)
plt.xlabel("Daily precipitation (mm)")
plt.ylabel("Daily mean NO₂ (µg/m³)")
plt.title("Precipitation and NO₂")
plt.show()

In [ ]:
# Simple correlations are descriptive only; they are not causal effects.
corr = data[["no2", "temp_mean", "precip", "wind_max"]].corr()
display(corr.round(3))

# Monthly means help show the seasonal structure
monthly = data.groupby(data["date"].dt.month).agg(
    mean_no2=("no2", "mean"),
    mean_temp=("temp_mean", "mean"),
    mean_wind=("wind_max", "mean")
)
display(monthly.round(2))

# Inspect the largest NO2 values instead of deleting them automatically
display(data.nlargest(10, "no2"))

### Exploratory interpretation

Temperature and wind both have negative simple associations with NO₂. Wind has the stronger correlation, while precipitation is much weaker. I also see several unusually high NO₂ days in late winter and early spring.

This motivates three models:

1. **Model 1:** temperature only, as a simple baseline;
2. **Model 2:** temperature + wind, because wind is physically related to pollutant dispersion and is strongly associated with NO₂ in the exploratory data;
3. **Model 3:** temperature + temperature² + wind, to test whether nonlinear temperature structure improves prediction.

I do not add precipitation to the main models just because it is available. Its simple association is weak, and adding predictors without a clear reason can make interpretation harder.

## 3. Train-Test Split and Standardization

In [ ]:
# The assignment asks for a random ~80/20 split.
train, test = train_test_split(
    data,
    test_size=0.20,
    random_state=RANDOM_SEED
)

train = train.copy()
test = test.copy()

print("Training observations:", len(train))
print("Test observations:", len(test))

In [ ]:
# IMPORTANT: means and standard deviations come from the TRAINING data only.
# The same training-set values are then used to transform the test data.
# This prevents information from the test set leaking into model fitting.

scaling = {}

for col in ["temp_mean", "wind_max", "precip"]:
    mean = train[col].mean()
    sd = train[col].std()
    scaling[col] = (mean, sd)

    train[col + "_std"] = (train[col] - mean) / sd
    test[col + "_std"] = (test[col] - mean) / sd

# Polynomial term for Model 3
train["temp_std2"] = train["temp_mean_std"] ** 2
test["temp_std2"] = test["temp_mean_std"] ** 2

display(pd.DataFrame(
    scaling,
    index=["training mean", "training sd"]
).T)

## 4. Bayesian Regression Models

All three models use a Normal likelihood for daily NO₂. The predictors are standardized, so a slope is the expected difference in NO₂ associated with a one-standard-deviation increase in that predictor, holding the other predictors fixed.

I use the same basic priors across models:

- \(a \sim \text{Normal}(25, 15)\)
- regression slopes \(b \sim \text{Normal}(0, 5)\)
- \(\sigma \sim \text{Uniform}(0, 30)\)

The slope prior is symmetric because I do not force the weather association to be positive or negative. The scale is broad relative to the effects I expect after standardizing the predictors.

### Model 1: Temperature only

\[
y_i \sim \text{Normal}(\mu_i, \sigma), \qquad
\mu_i = a + b_T T_i
\]

In [ ]:
with pm.Model() as model_1:
    # Priors
    a = pm.Normal("a", mu=25, sigma=15)
    b_temp = pm.Normal("b_temp", mu=0, sigma=5)
    sigma = pm.Uniform("sigma", lower=0, upper=30)

    # Training predictor
    x_temp = pm.Data("x_temp", train["temp_mean_std"].values)

    # Linear regression mean
    mu = pm.Deterministic("mu", a + b_temp * x_temp)

    # Likelihood
    pm.Normal(
        "likelihood",
        mu=mu,
        sigma=sigma,
        observed=train["no2"].values
    )

    # Prior-predictive and posterior samples
    prior_1 = pm.sample_prior_predictive(
        samples=500,
        random_seed=RANDOM_SEED
    )

    idata_1 = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        cores=2,
        target_accept=0.9,
        random_seed=RANDOM_SEED
    )

### Model 2: Temperature + wind

\[
\mu_i = a + b_T T_i + b_W W_i
\]

In [ ]:
with pm.Model() as model_2:
    a = pm.Normal("a", mu=25, sigma=15)
    b_temp = pm.Normal("b_temp", mu=0, sigma=5)
    b_wind = pm.Normal("b_wind", mu=0, sigma=5)
    sigma = pm.Uniform("sigma", lower=0, upper=30)

    x_temp = pm.Data("x_temp", train["temp_mean_std"].values)
    x_wind = pm.Data("x_wind", train["wind_max_std"].values)

    mu = pm.Deterministic(
        "mu",
        a + b_temp * x_temp + b_wind * x_wind
    )

    pm.Normal(
        "likelihood",
        mu=mu,
        sigma=sigma,
        observed=train["no2"].values
    )

    prior_2 = pm.sample_prior_predictive(
        samples=500,
        random_seed=RANDOM_SEED
    )

    idata_2 = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        cores=2,
        target_accept=0.9,
        random_seed=RANDOM_SEED
    )

### Model 3: Quadratic temperature + wind

\[
\mu_i = a + b_T T_i + b_{T^2}T_i^2 + b_W W_i
\]

This is still a linear regression model because it is linear in the coefficients, even though the fitted relationship can curve as a function of temperature. This follows the polynomial-regression approach from Session 9.

In [ ]:
with pm.Model() as model_3:
    a = pm.Normal("a", mu=25, sigma=15)
    b_temp = pm.Normal("b_temp", mu=0, sigma=5)
    b_temp2 = pm.Normal("b_temp2", mu=0, sigma=5)
    b_wind = pm.Normal("b_wind", mu=0, sigma=5)
    sigma = pm.Uniform("sigma", lower=0, upper=30)

    x_temp = pm.Data("x_temp", train["temp_mean_std"].values)
    x_temp2 = pm.Data("x_temp2", train["temp_std2"].values)
    x_wind = pm.Data("x_wind", train["wind_max_std"].values)

    mu = pm.Deterministic(
        "mu",
        a
        + b_temp * x_temp
        + b_temp2 * x_temp2
        + b_wind * x_wind
    )

    pm.Normal(
        "likelihood",
        mu=mu,
        sigma=sigma,
        observed=train["no2"].values
    )

    prior_3 = pm.sample_prior_predictive(
        samples=500,
        random_seed=RANDOM_SEED
    )

    idata_3 = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        cores=2,
        target_accept=0.9,
        random_seed=RANDOM_SEED
    )

## 5. Prior, Posterior, and Posterior-Predictive Checks

### Prior-predictive check

Session 7 emphasizes checking what the model can generate before seeing the data. I compare the Model 2 prior-predictive distribution with the observed training data. I expect the prior prediction to be much broader than the data, but it should still put substantial probability on NO₂ values in the observed range.

In [ ]:
# Prior-predictive check for Model 2
prior_y = prior_2.prior_predictive["likelihood"].values.ravel()

plt.figure(figsize=(7, 4))
plt.hist(
    prior_y,
    bins=60,
    density=True,
    alpha=0.45,
    label="Prior predictive"
)
plt.hist(
    train["no2"],
    bins=25,
    density=True,
    alpha=0.60,
    label="Observed training data"
)
plt.xlabel("Daily mean NO₂ (µg/m³)")
plt.ylabel("Density")
plt.title("Prior-predictive check for Model 2")
plt.legend()
plt.show()

print("Prior predictive 5th and 95th percentiles:",
      np.quantile(prior_y, [0.05, 0.95]))

Because the likelihood is Normal, the prior predictive distribution can include some negative NO₂ values, which are not physically possible. I treat this as a limitation of the Gaussian approximation rather than evidence that negative concentrations are expected. The prior remains intentionally broad and does not force the posterior toward a particular weather effect.

In [ ]:
# Helper function for sampler diagnostics
def sampler_diagnostics(idata, var_names):
    summary = az.summary(idata, var_names=var_names, round_to=2)
    divergences = int(idata.sample_stats["diverging"].sum().values)

    display(summary)
    print("Divergences:", divergences)

    az.plot_rank(idata, var_names=var_names)
    plt.show()

    return summary

In [ ]:
print("MODEL 1 DIAGNOSTICS")
summary_1 = sampler_diagnostics(
    idata_1,
    ["a", "b_temp", "sigma"]
)

In [ ]:
print("MODEL 2 DIAGNOSTICS")
summary_2 = sampler_diagnostics(
    idata_2,
    ["a", "b_temp", "b_wind", "sigma"]
)

In [ ]:
print("MODEL 3 DIAGNOSTICS")
summary_3 = sampler_diagnostics(
    idata_3,
    ["a", "b_temp", "b_temp2", "b_wind", "sigma"]
)

For a satisfactory sampler run I look for R-hat values close to 1, effective sample sizes comfortably above a few hundred, approximately uniform rank plots, and no divergences. These are the same diagnostics emphasized in Sessions 7 and 8.

In [ ]:
# Posterior distributions for the weather coefficients in the two more complete models
az.plot_posterior(
    idata_2,
    var_names=["b_temp", "b_wind"],
    hdi_prob=0.89
)
plt.show()

az.plot_posterior(
    idata_3,
    var_names=["b_temp", "b_temp2", "b_wind"],
    hdi_prob=0.89
)
plt.show()

### Posterior-predictive checks

In [ ]:
# Add posterior-predictive samples to each InferenceData object
with model_1:
    pm.sample_posterior_predictive(
        idata_1,
        var_names=["likelihood"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED
    )

with model_2:
    pm.sample_posterior_predictive(
        idata_2,
        var_names=["likelihood"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED
    )

with model_3:
    pm.sample_posterior_predictive(
        idata_3,
        var_names=["likelihood"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED
    )

In [ ]:
# Compare observed training data with replicated data from the fitted models
for label, idata in [
    ("Model 1", idata_1),
    ("Model 2", idata_2),
    ("Model 3", idata_3),
]:
    az.plot_ppc(idata, num_pp_samples=100)
    plt.title(f"{label}: posterior-predictive check")
    plt.show()

The posterior-predictive checks should reproduce the center and general spread of the observed distribution. A feature to look for is whether the Normal likelihood has difficulty reproducing the largest pollution spikes. Session 8 showed why a Student-t likelihood can be more robust when a few observations have heavier tails than a Normal model expects.

## 6. Model Comparison

The assignment requires the models to be compared on the held-out 20% test set. I use posterior samples to make predictions for the test predictors, and then calculate:

- **RMSE**, which penalizes large errors more strongly;
- **MAE**, which is the average absolute prediction error.

Lower values are better for both metrics.

In [ ]:
def test_posterior_predictive(idata, model_number, test_df, seed=RANDOM_SEED):
    # Stack chain and draw into one posterior-sample dimension
    post = idata.posterior.stack(sample=("chain", "draw"))

    a = post["a"].values
    sigma = post["sigma"].values

    # Compute mu for every test observation and every posterior draw
    if model_number == 1:
        b_temp = post["b_temp"].values
        mu_samples = (
            a[None, :]
            + test_df["temp_mean_std"].values[:, None] * b_temp[None, :]
        )

    elif model_number == 2:
        b_temp = post["b_temp"].values
        b_wind = post["b_wind"].values
        mu_samples = (
            a[None, :]
            + test_df["temp_mean_std"].values[:, None] * b_temp[None, :]
            + test_df["wind_max_std"].values[:, None] * b_wind[None, :]
        )

    elif model_number == 3:
        b_temp = post["b_temp"].values
        b_temp2 = post["b_temp2"].values
        b_wind = post["b_wind"].values
        mu_samples = (
            a[None, :]
            + test_df["temp_mean_std"].values[:, None] * b_temp[None, :]
            + test_df["temp_std2"].values[:, None] * b_temp2[None, :]
            + test_df["wind_max_std"].values[:, None] * b_wind[None, :]
        )

    else:
        raise ValueError("model_number must be 1, 2, or 3")

    # Posterior predictive: add observation noise to each posterior mean
    local_rng = np.random.default_rng(seed)
    y_pred_samples = local_rng.normal(
        loc=mu_samples,
        scale=sigma[None, :]
    )

    pred_mean = y_pred_samples.mean(axis=1)
    pred_lower = np.quantile(y_pred_samples, 0.055, axis=1)
    pred_upper = np.quantile(y_pred_samples, 0.945, axis=1)

    return y_pred_samples, pred_mean, pred_lower, pred_upper

In [ ]:
pred_samples_1, pred_mean_1, lower_1, upper_1 = test_posterior_predictive(
    idata_1, 1, test
)

pred_samples_2, pred_mean_2, lower_2, upper_2 = test_posterior_predictive(
    idata_2, 2, test
)

pred_samples_3, pred_mean_3, lower_3, upper_3 = test_posterior_predictive(
    idata_3, 3, test
)

In [ ]:
# Test-set RMSE and MAE
results = pd.DataFrame({
    "Model": [
        "M1: temperature",
        "M2: temperature + wind",
        "M3: quadratic temperature + wind"
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(test["no2"], pred_mean_1)),
        np.sqrt(mean_squared_error(test["no2"], pred_mean_2)),
        np.sqrt(mean_squared_error(test["no2"], pred_mean_3))
    ],
    "MAE": [
        mean_absolute_error(test["no2"], pred_mean_1),
        mean_absolute_error(test["no2"], pred_mean_2),
        mean_absolute_error(test["no2"], pred_mean_3)
    ]
})

display(results.round(2))

In [ ]:
# Visual model comparison
plt.figure(figsize=(7, 4))
plt.bar(results["Model"], results["RMSE"])
plt.ylabel("Test RMSE (µg/m³)")
plt.title("Held-out prediction error")
plt.xticks(rotation=18, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Observed vs predicted for Model 2 (the model expected to perform best)
plt.figure(figsize=(6, 5))
plt.scatter(test["no2"], pred_mean_2, alpha=0.7)

minimum = min(test["no2"].min(), pred_mean_2.min())
maximum = max(test["no2"].max(), pred_mean_2.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Observed NO₂ (µg/m³)")
plt.ylabel("Posterior predictive mean NO₂ (µg/m³)")
plt.title("Model 2: observed vs predicted on test data")
plt.show()

In [ ]:
# Prediction intervals for Model 2 on held-out days
test_results = test[["date", "no2"]].copy()
test_results["pred_mean"] = pred_mean_2
test_results["pred_lower"] = lower_2
test_results["pred_upper"] = upper_2
test_results = test_results.sort_values("date")

plt.figure(figsize=(11, 4))
plt.scatter(
    test_results["date"],
    test_results["no2"],
    label="Observed"
)
plt.scatter(
    test_results["date"],
    test_results["pred_mean"],
    label="Posterior predictive mean"
)
plt.vlines(
    test_results["date"],
    test_results["pred_lower"],
    test_results["pred_upper"],
    alpha=0.25
)

plt.xlabel("Date")
plt.ylabel("NO₂ (µg/m³)")
plt.title("Model 2 predictions on held-out test days")
plt.legend()
plt.show()

### Model-selection interpretation

The best model is the one with the lowest held-out prediction error, not automatically the most flexible model. This is the main idea from Session 10: extra parameters can improve training fit without improving generalization.

After running the notebook, I use the table above to report the final RMSE/MAE values in the PDF.

## 7. Discussion and Limitations

Important limitations to keep in mind:

- The weather values are reanalysis estimates over a grid, while NO₂ is measured at one street-level station.
- Maximum daily wind speed is not the same as mean wind speed.
- Traffic volume, weekdays/weekends, heating, boundary-layer height, construction, and wind direction are not in the model.
- The assignment asks for a random train-test split, but daily observations are a time series. A future forecasting study should use a chronological or blocked split.
- The largest NO₂ values may be heavier-tailed than a Normal likelihood expects. A Student-t regression, as studied in Session 8, would be a logical extension.

## 8. Conclusion

The final conclusion should be based on the held-out test metrics. In the current analysis, adding wind to temperature improves prediction clearly, while the quadratic temperature term adds complexity without improving test performance. That makes the temperature-plus-wind model the best balance between predictive performance and simplicity among the models tested.

## 9. References

- Berliner Luftgütemessnetz. Station 221 Karl-Marx-Straße. https://luftdaten.berlin.de/station/mc221
- Open-Meteo. Historical Weather API. https://open-meteo.com/en/docs/historical-weather-api
- CS146 Session 7. `07a. Pre-class work.ipynb`, *Linear regression*.
- CS146 Session 8. `08a. Pre-class work.ipynb`, *Robust linear regression: An exploration*.
- CS146 Session 9. `09a. Pre-class work.ipynb`, *Linear regression for non-linear data*.
- CS146 Session 10. `10a. Pre-class work.ipynb`, *Model comparison*.

## Appendix: AI use

I used Gemini to brainstorm possible Berlin data sets at the beginning of the project. I also used ChatGPT to help debug PyMC code, compare my draft against the assignment grading guide, and suggest ways to organize and explain the analysis. I reviewed the code and report and made the final choices about the models, interpretation, and wording.